In [ ]:
!pip install -q \
transformers \
accelerate \
bitsandbytes \
sentence-transformers \
langgraph \
langchain \
chromadb \
pydantic

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/HuggingFaceModels"
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
import os

IMAGE_FOLDER = "/content/drive/MyDrive/MemoryLens/images"

os.makedirs(
    IMAGE_FOLDER,
    exist_ok=True
)

In [ ]:
import uuid

def save_image(image):

    image_id = str(uuid.uuid4())

    extension = image.format.lower() if image.format else "png"

    image_path = os.path.join(
    IMAGE_FOLDER,
    f"{image_id}.{extension}")


    image.save(image_path)

    return image_id, image_path

In [ ]:
from google.colab import files
from PIL import Image

# Upload image
uploaded = files.upload()

# Get uploaded file name
image_name = next(iter(uploaded))

# Open image
image = Image.open(image_name)

# Save image using your function
image_id, image_path = save_image(image)

print(image_id)
print(image_path)

In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

from sentence_transformers import SentenceTransformer

MODEL_NAME = "Qwen/Qwen2.5-1.8B-Instruct" # Changed to a smaller model to fit GPU memory





In [ ]:
import os

MODEL_CACHE = "/content/drive/MyDrive/HuggingFaceModels/models--Qwen--Qwen2.5-VL-7B-Instruct"

for root, dirs, files in os.walk(MODEL_CACHE):
    if "config.json" in files:
        print(root)

/content/drive/MyDrive/HuggingFaceModels/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/HuggingFaceModels/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5"



In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

vision_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_PATH,)

In [ ]:
embedding_model=SentenceTransformer(
    "BAAI/bge-base-en-v1.5"

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
print(vision_model.get_memory_footprint()/1024**3, "GB")

5.387335374951363 GB


In [ ]:
def ask_vlm(image,prompt):
  messages = [
      {
          "role":"user",
          "content":[
              {
                  "type":"image",
                  "image":image
              },
              {
                  "type":"text",
                  "text":prompt
              }
          ]
      }
  ]


  text=processor.apply_chat_template(
      messages,
      tokenize=False,

      add_generation_prompt=True
  )

  inputs=processor(
      text=text,
      images=[image], # Wrap the image in a list here
      return_tensors="pt",

  ).to(vision_model.device)


  with torch.no_grad(): #Uses less GPU memory.Slightly faster.Standard practice for inference.

      output=vision_model.generate(**inputs,
                               max_new_tokens=512
                               )


  generated=output[:,inputs["input_ids"].shape[1]:
                   ]

  response=processor.batch_decode(
      generated,
      skip_special_tokens=True,

  )[0]

  return response

In [ ]:
from pydantic import BaseModel


class VisionMetadata(BaseModel):

    application: str

    primary_subject: str

    category: str

    contains_code: bool

    contains_diagram: bool

    language: str

    importance:str

    searchable_tags:list[str]

In [ ]:
from pydantic import BaseModel
from typing import Optional


class ScreenshotMetadata(BaseModel):

    image_id: str

    image_path: str

    ocr_text: Optional[str] = None

    vision_metadata: Optional[VisionMetadata] = None

    extracted_memory: Optional[str] = None

In [ ]:
from typing import TypedDict
from PIL import Image


class MemoryLensState(TypedDict):

    image: Image.Image

    image_id: str

    image_path: str

    metadata: ScreenshotMetadata

    retrieved_documents: list[str]

    retrieved_metadata: list[dict]

    retrieved_image_paths: list[str]


    query: str

    final_answer: str

In [ ]:
OCR_PROMPT = """
Extract every visible text from this image.

Rules:
1. Preserve the original wording.
2. Preserve the reading order.
3. Do not summarize.
4. If no text exists, return "No text found."
"""

In [ ]:
def ocr_agent(state: MemoryLensState):

  image=state["image"]

  ocr_text=ask_vlm(
      image=image,
      prompt=OCR_PROMPT
  )

  state["metadata"].ocr_text=ocr_text
  return state

In [ ]:
VISION_PROMPT = """
You are an expert vision-language assistant.

Analyze the given image carefully and return ONLY valid JSON.

Rules:
1. Return ONLY valid JSON.
2. Do NOT include explanations.
3. Do NOT use markdown.
4. If a field cannot be determined, use "Unknown".
5. Boolean fields must be either true or false.
6. searchable_tags must contain 3 to 8 relevant tags.

Return JSON in this exact format:

{
    "application": "",
    "primary_subject": "",
    "category": "",
    "contains_text": false,
    "contains_code": false,
    "contains_diagram": false,
    "language": "",
    "importance": "",
    "searchable_tags": []
}

Field descriptions:

application:
If the image is a screenshot, identify the application or website.
Examples:
ChatGPT
Google Chrome
VS Code
WhatsApp
Instagram
YouTube
Google Colab
LeetCode
Amazon
Flipkart
Unknown

If it is NOT a screenshot, return "None".

------------------------------------------------

primary_subject:
The main object, person, place, product, document or topic in the image.

Examples:
Coca-Cola Bottle
Pepsi Can
Python Programming
Restaurant Menu
Passport
Driving License
Laptop
Invoice
Dog
Cat
Mountain
Unknown

------------------------------------------------

category:
Choose ONLY ONE.

Document
Screenshot
Product
Food
Person
Vehicle
Animal
Nature
Building
Electronics
Clothing
Medical
Finance
Education
Shopping
Receipt
Certificate
Identity Card
Artwork
Other

------------------------------------------------

contains_text:
true if readable text exists.

------------------------------------------------

contains_code:
true if programming code is visible.

------------------------------------------------

contains_diagram:
true if flowcharts, graphs, mind maps, architecture diagrams, UML diagrams or similar visuals are present.

------------------------------------------------

language:
Primary language visible in the image.

Examples:
English
Tamil
Hindi
French
Mixed
Unknown

------------------------------------------------

importance:

Choose ONLY ONE.

HIGH
MEDIUM
LOW

Assign importance based on future usefulness.

HIGH
- Important documents
- IDs
- Receipts
- Bills
- Certificates
- Medical records
- Educational notes
- Programming screenshots
- Important conversations

MEDIUM
- Shopping
- Products
- Food
- General screenshots
- Normal conversations

LOW
- Memes
- Random images
- Wallpapers
- Temporary content
- Low-information images

------------------------------------------------

searchable_tags:

Generate 3 to 8 concise tags.

Rules:
- lowercase
- no duplicates
- use single words where possible
- use underscore instead of spaces
- include important objects, brands, technologies, applications and topics
- avoid generic words like image, photo, screenshot

Examples:

["cocacola","bottle","drink"]

["python","langgraph","rag","chatgpt"]

["passport","identity","government"]

["amazon","shopping","headphones"]

["invoice","payment","finance"]
"""

In [ ]:
import json
import re

def vision_agent(state: MemoryLensState):

    image = state["image"]

    response = ask_vlm(
        image=image,
        prompt=VISION_PROMPT
    )

    # Extract only the JSON object
    match = re.search(r"\{.*\}", response, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON found:\n{response}")

    data = json.loads(match.group())

    metadata = VisionMetadata.model_validate(data)

    state["metadata"].vision_metadata = metadata

    return state

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM

)

text_model_name = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
import os

MODEL_CACHE = "/content/drive/MyDrive/HuggingFaceModels/models--Qwen--Qwen2.5-3B-Instruct"

for root, dirs, files in os.walk(MODEL_CACHE):
    if "config.json" in files:
        print(root)

/content/drive/MyDrive/HuggingFaceModels/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1


In [ ]:
llm_model_path = "/content/drive/MyDrive/HuggingFaceModels/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1"

In [ ]:
text_tokenizer=AutoTokenizer.from_pretrained(
    llm_model_path,

)

text_model=AutoModelForCausalLM.from_pretrained(
    llm_model_path,
    quantization_config=bnb_config,
    device_map="cpu",

)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
MEMORY_PROMPT = """
You are an AI knowledge extraction assistant.

Your task is to convert the image information into a concise,
searchable memory.

You will receive:

1. OCR text extracted from the image.

2. Structured metadata describing the image.

Your job is to identify the important information that should
be remembered for future retrieval.

Rules:

1. Keep only useful information.
2. Remove greetings and unnecessary conversation.
3. Preserve important facts.
4. If the image contains learning material, summarize the concepts.
5. If the image is a document, summarize the document.
6. If the image is a receipt, summarize the purchase.
7. If the image is a product, describe the product.
8. If nothing useful exists, return
"No meaningful memory found."

Return only the extracted memory.
"""

In [ ]:
def knowledege_extraction_agent(state:MemoryLensState):

  ocr = state["metadata"].ocr_text

  vision=state["metadata"].vision_metadata

  prompt=  f""" OCR TEXT: {ocr}
  VISION METADATA:{vision.model_dump_json(indent=2)}

  {MEMORY_PROMPT}
  """

  memory=ask_llm(prompt)
  state["metadata"].extracted_memory=memory

  return state






In [ ]:
def ask_llm(prompt:str)-> str :

  messages=[
      {
          "role":"user",
          "content":prompt
      }
  ]

  text=text_tokenizer.apply_chat_template(
      messages,
      tokenize=False,

      add_generation_prompt=True
  )


  inputs=text_tokenizer(text=text,return_tensors="pt").to(text_model.device)

  with torch.no_grad():
    outputs=text_model.generate(**inputs,
                                max_new_tokens=512)

  generated=outputs[:,inputs["input_ids"].shape[1]:]

  response=text_tokenizer.batch_decode(
      generated,
      skip_special_tokens=True
  )[0]

  return response




In [ ]:
def build_search_document(
    metadata: ScreenshotMetadata
) -> str:

    vision = metadata.vision_metadata

    document = f"""
OCR TEXT

{metadata.ocr_text}

--------------------------------

EXTRACTED MEMORY

{metadata.extracted_memory}

--------------------------------

APPLICATION

{vision.application}

--------------------------------

PRIMARY SUBJECT

{vision.primary_subject}

--------------------------------

CATEGORY

{vision.category}

--------------------------------

SEARCHABLE TAGS

{", ".join(vision.searchable_tags)}

--------------------------------

IMPORTANCE

{vision.importance}
"""

    return document

In [ ]:
def store_screenshot(

                     metadata:ScreenshotMetadata,
                     ):
  document=build_search_document(metadata)

  embedding=embedding_model.encode(
      document
  ).tolist()#the embedding returned is a numpy array but chromaDb expects a normal python list

  metadata_dict = {
        "image_path": metadata.image_path,
        "application": metadata.vision_metadata.application,
        "category": metadata.vision_metadata.category,
        "importance": metadata.vision_metadata.importance,
        "language": metadata.vision_metadata.language
    }

  collection.add(
      ids=[metadata.image_id],
      documents=[document],
      embeddings=[embedding],
      metadatas=[metadata_dict]
  )






In [ ]:
import chromadb

In [ ]:
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/MemoryLens/chromadb"
)

In [ ]:
collection = client.get_or_create_collection(
    name="memorylens"
)

In [ ]:
def retrieve_documents(query:str):

  query_embedding=embedding_model.encode(
      query
  ).tolist()

  results=collection.query(
      query_embeddings=[query_embedding],
      n_results=5
  )

  return results

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-electra-base')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def rerank_documents(

                     query:str,
                     retrieve_documents: list[str]
                     ):
  pairs=[(query,document) for document in retrieve_documents]

  scores=reranker.predict(pairs)



  ranked_documents=sorted(
      zip(scores,retrieve_documents),
      reverse=True
  )

  return [
      document
      for score,document in ranked_documents[:5]
  ]


In [ ]:
REASONING_PROMPT = """
You are MemoryLens.

The retrieved screenshots are shown below.

Your task:

1. Find every screenshot that matches the user's question.
2. Ignore screenshots that are unrelated.
3. If one or more screenshots match, answer using ONLY those screenshots.
4. Never say the information is missing if it exists in any screenshot.
5. If no screenshot matches, reply exactly:
I couldn't find that information in your screenshots.

Retrieved Screenshots:

{context}

User Query:
{query}

Answer:
"""

In [ ]:
def memory_reasoning_agent(
    query: str,
    retrieved_documents: list[str]
):

    context = ""

    for i, doc in enumerate(retrieved_documents, 1):
        context += f"""
========== Screenshot {i} ==========

{doc}

"""

    prompt = REASONING_PROMPT.format(
        context=context,
        query=query
    )

    answer = ask_llm(prompt)

    return answer

In [ ]:
!pip install -q langgraph

In [ ]:
from langgraph.graph import StateGraph
from langgraph.graph import START, END

In [ ]:
graph = StateGraph(
    MemoryLensState
)

In [ ]:
graph.add_node("ocr", ocr_agent)
graph.add_node("vision", vision_agent)
graph.add_node("knowledge_extraction", knowledege_extraction_agent)

graph.add_edge(START, "ocr")
graph.add_edge("ocr", "vision")
graph.add_edge("vision", "knowledge_extraction")
graph.add_edge("knowledge_extraction", END)

memory_graph = graph.compile()

In [ ]:
retrieval_graph = StateGraph(MemoryLensState)

In [ ]:
def retrieve_node(state: MemoryLensState):

    # Create query embedding
    query_embedding = embedding_model.encode(
        state["query"]
    ).tolist()


    # Vector search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10
    )


    documents = results["documents"][0]
    metadatas = results["metadatas"][0]


    # Reranking
    pairs = [
        (state["query"], doc)
        for doc in documents
    ]

    scores = reranker.predict(pairs)


    # Sort based on reranker score
    ranked = sorted(
        zip(scores, documents, metadatas),
        key=lambda x: x[0],
        reverse=True
    )


    # Store top 3
    state["retrieved_documents"] = [
        doc
        for _, doc, _ in ranked[:3]
    ]


    state["retrieved_metadata"] = [
        metadata
        for _, _, metadata in ranked[:3]
    ]


    # Extract image paths
    state["retrieved_image_paths"] = [
        metadata["image_path"]
        for _, _, metadata in ranked[:3]
    ]


    return state

In [ ]:
retrieval_graph = StateGraph(MemoryLensState)

def reasoning_node(state: MemoryLensState):
    query = state["query"]
    retrieved_documents = state["retrieved_documents"]
    answer = memory_reasoning_agent(query, retrieved_documents)
    state["final_answer"] = answer
    return state

retrieval_graph.add_node("retrieve", retrieve_node)
retrieval_graph.add_node("reason", reasoning_node)

retrieval_graph.add_edge(START, "retrieve")
retrieval_graph.add_edge("retrieve", "reason")
retrieval_graph.add_edge("reason", END)

retrieval_graph = retrieval_graph.compile()

In [ ]:
import os
from PIL import Image

IMAGE_FOLDER = "/content/drive/MyDrive/MemoryLens/images"

# Get all IDs already stored in ChromaDB
stored_ids = set(collection.get()["ids"])

print(f"Already stored: {len(stored_ids)} images")

for filename in os.listdir(IMAGE_FOLDER):

    if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    image_id = os.path.splitext(filename)[0]

    # Skip if already stored
    if image_id in stored_ids:
        print(f"⏩ Skipped: {filename}")
        continue

    image_path = os.path.join(IMAGE_FOLDER, filename)
    image = Image.open(image_path)

    state = {
        "image": image,
        "image_id": image_id,
        "image_path": image_path,
        "metadata": ScreenshotMetadata(
            image_id=image_id,
            image_path=image_path
        ),
        "retrieved_documents": [],
        "query": "",
        "final_answer": ""
    }

    result = memory_graph.invoke(state)

    store_screenshot(result["metadata"])

    print(f"✅ Stored: {filename}")

In [ ]:
state = {
    "image": None,
    "image_id": "",
    "image_path": "",
    "metadata": None,
    "retrieved_documents": [],
    "retrieved_metadata": [], # Initialize missing key
    "retrieved_image_paths": [], # Initialize missing key
    "query": "give me 2 questions from the sql screenshot",
    "final_answer": ""
}

result = retrieval_graph.invoke(state)

answer = result["final_answer"]

answer = answer.replace("**", "")
answer = answer.replace("###", "")



print(answer)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os


for path in result["retrieved_image_paths"]:

    image = Image.open(path)

    plt.figure(figsize=(5,5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(os.path.basename(path))
    plt.show()